In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv("../data/reviews.csv")

df['Sentiment'] = df['Rating'].apply(lambda x: 1 if x >= 4 else 0)

print("✅ Data successfully loaded")
print("Initial Sentiment Distribution:")
print(df['Sentiment'].value_counts())

✅ Data successfully loaded
Initial Sentiment Distribution:
Sentiment
0    304
1    196
Name: count, dtype: int64


In [4]:
def clean_text(text):
    
    text = text.lower()
    
    text = re.sub(r'[^\w\s]', '', text)
    
    text = text.strip()
    return text


df['Cleaned_Review'] = df['Review_Text'].apply(clean_text)


df[['Review_Text', 'Cleaned_Review']].head()

,Review_Text,Cleaned_Review
0,The actual product does not match the premium ...,the actual product does not match the premium ...
1,"Value for money. Build quality is solid, but p...",value for money build quality is solid but pac...
2,Very nice features and easy to set up. Happy w...,very nice features and easy to set up happy wi...
3,Average product. It does the job but nothing e...,average product it does the job but nothing ex...
4,Absolutely amazing! The sound quality is cryst...,absolutely amazing the sound quality is crysta...


In [5]:
X = df['Cleaned_Review']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Total training reviews: {len(X_train)}")
print(f"Total testing reviews: {len(X_test)}")

Total training reviews: 400
Total testing reviews: 100


In [6]:
# 1.  TF-IDF initializarion for coverting text into numbers
vectorizer = TfidfVectorizer(max_features=2000)

# Transforming Train and Test Data
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 2. Logistic Regression model initialization and train 
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

# 3. for checking model accuracy
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"🔥 Model successfully trained !")
print(f"🎯 Test Accuracy: {accuracy * 100:.2f}%")

🔥 Model successfully trained !
🎯 Test Accuracy: 100.00%


In [7]:
print("--- 📊 Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

--- 📊 Classification Report ---
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        61
    Positive       1.00      1.00      1.00        39

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [8]:
def predict_sentiment(custom_review):
    # 1. cleaning text
    cleaned = clean_text(custom_review)
    # 2. Vectorizarion
    vectorized = vectorizer.transform([cleaned])
    # 3.Prediction
    prediction = model.predict(vectorized)[0]

    sentiment = "Positive " if prediction == 1 else "Negative"
    print(f"Review: '{custom_review}")
    print(f"Predicted Sentiment: {sentiment}\n")


predict_sentiment("This product is absolutely amzing, loved the quality!")
predict_sentiment("Worst purchase ever. Total  waste of money and broke in two days.")

Review: 'This product is absolutely amzing, loved the quality!
Predicted Sentiment: Negative

Review: 'Worst purchase ever. Total  waste of money and broke in two days.
Predicted Sentiment: Negative



In [9]:
print(y_train.value_counts())


Sentiment
0    243
1    157
Name: count, dtype: int64


In [10]:
# 1. Advanced TF-IDF Vectorizer with Bi-grams setup 
# ngram_range=(1, 2) means it learn singel and double words 
upgraded_vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))

# Data ko transform karna
X_train_upgraded = upgraded_vectorizer.fit_transform(X_train)
X_test_upgraded = upgraded_vectorizer.transform(X_test)

# 2. Logistic Regression  regularize (C=1.0) and train again
upgraded_model = LogisticRegression(C=1.0, class_weight='balanced')
upgraded_model.fit(X_train_upgraded, y_train)

# 3. Checking New accuracy
y_pred_upgraded = upgraded_model.predict(X_test_upgraded)
print(f"🎯 Upgraded Model Test Accuracy: {accuracy_score(y_test, y_pred_upgraded) * 100:.2f}%")

🎯 Upgraded Model Test Accuracy: 100.00%


In [11]:
def predict_sentiment_v2(custom_review):

    cleaned = clean_text(custom_review)

    vectorized = upgraded_vectorizer.transform([cleaned])

    prediction = upgraded_model.predict(vectorized)[0]

    sentiment = "Positive " if prediction == 1 else "Negative "
    print(f"Review: '{custom_review}'")
    print(f"Predicted Sentiment: {sentiment}\n")

predict_sentiment_v2("This product is absolutely amzing, loved the quality!")
predict_sentiment_v2("Worst purchase ever. Total waste of money and broke in two days.")

Review: 'This product is absolutely amzing, loved the quality!'
Predicted Sentiment: Negative 

Review: 'Worst purchase ever. Total waste of money and broke in two days.'
Predicted Sentiment: Negative 



In [12]:
print("--- Original Target Labels Distribution ---")
print(df['Sentiment'].value_counts())

--- Original Target Labels Distribution ---
Sentiment
0    304
1    196
Name: count, dtype: int64


In [ ]:
# Check some sample feature names from the vocabulary
feature_names = upgraded_vectorizer.get_feature_names_out()
print("Total Features Extracted:", len(feature_names))
print("Sample Features:", feature_names[:30])